In [1]:
import pandas as pd

# URL of the Canada immigration dataset in CSV format
csv_url_canada = "https://docs.google.com/spreadsheets/d/1OGs8KoHGEZDxyFdaxAKmZbeO1BiSWVQ6/export?format=csv&gid=361756276"

#pandas is used for loading and manipulating tabular data.
#The dataset comes from a Google Sheets CSV export link


In [2]:
def load_canada_detailed_clean(url):
    # Load CSV without headers (data has extra metadata on top)
    df = pd.read_csv(url, header=None, dtype=str)
    
    # Skip top metadata rows (first 20 rows)
    df = df.iloc[20:].reset_index(drop=True)
    
    # Define fixed columns (first 9) and years (1980-2013)
    fixed_cols = ["Type", "Coverage", "OdName", "AREA", "AreaName", "REG", "RegName", "DEV", "DevName"]
    years = [str(y) for y in range(1980, 2014)]
    
    # Assign column names
    df.columns = fixed_cols + years
    
    # Drop rows where all year columns are empty
    df = df.dropna(how='all')
    
    # Skip first row (possibly redundant headers)
    df = df.iloc[1:].reset_index(drop=True)
    
    # Forward-fill metadata columns to propagate country/region names
    df[["Type", "Coverage", "OdName", "AreaName", "RegName", "DevName"]] = \
        df[["Type", "Coverage", "OdName", "AreaName", "RegName", "DevName"]].ffill()
    
    # Convert year columns to numeric, fill missing with 0, convert to int
    for col in years:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    
    return df
#The dataset contains metadata rows, so we skip the first 20 rows.
#Column names include metadata columns + years 1980–2013.
#Forward-fill (ffill) ensures that countries and regions are properly labeled for all rows.
#Yearly immigration numbers are converted to integer type for calculations.


In [3]:
# Load cleaned dataset
df_canada_clean = load_canada_detailed_clean(csv_url_canada)

# Display first few rows to inspect data
df_canada_clean.head()


#This shows the initial cleaned dataset.
#You should now see OdName (country), Area/Region/DevName, and columns for each year 1980–2013.


,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,...,1450,1223,856,702,560,716,561,539,620,603
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,Immigrants,Foreigners,American Samoa,909,Oceania,957,Polynesia,902,Developing regions,0,...,0,0,1,0,0,0,0,0,0,0
4,Immigrants,Foreigners,Andorra,908,Europe,925,Southern Europe,901,Developed regions,0,...,0,0,1,1,0,0,0,0,1,1


In [4]:
# Sort dataset by country name (OdName)
df_sorted = df_canada_clean.sort_values("OdName").reset_index(drop=True)

# Display first few rows after sorting
df_sorted.head()

#Sorting by column OdName makes it easier to find and compare countries.
#reset_index(drop=True) ensures a clean index.


,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,...,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,...,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,...,1450,1223,856,702,560,716,561,539,620,603
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,...,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331
3,Immigrants,Foreigners,American Samoa,909,Oceania,957,Polynesia,902,Developing regions,0,...,0,0,1,0,0,0,0,0,0,0
4,Immigrants,Foreigners,Andorra,908,Europe,925,Southern Europe,901,Developed regions,0,...,0,0,1,1,0,0,0,0,1,1


In [5]:
# Define list of year columns
year_cols = [str(y) for y in range(1980, 2014)]

# Compute total immigration per country
df_sorted["Total"] = df_sorted[year_cols].sum(axis=1)

# Preview first few rows with total
df_sorted.head()

#Adds a Total column summing all years 1980–2013 for each country.
#Useful for sorting or ranking countries by total immigration.


,Type,Coverage,OdName,AREA,AreaName,REG,RegName,DEV,DevName,1980,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
0,Immigrants,Foreigners,Afghanistan,935,Asia,5501,Southern Asia,902,Developing regions,16,...,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
1,Immigrants,Foreigners,Albania,908,Europe,925,Southern Europe,901,Developed regions,1,...,1223,856,702,560,716,561,539,620,603,15699
2,Immigrants,Foreigners,Algeria,903,Africa,912,Northern Africa,902,Developing regions,80,...,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
3,Immigrants,Foreigners,American Samoa,909,Oceania,957,Polynesia,902,Developing regions,0,...,0,1,0,0,0,0,0,0,0,6
4,Immigrants,Foreigners,Andorra,908,Europe,925,Southern Europe,901,Developed regions,0,...,0,1,1,0,0,0,0,1,1,15


In [6]:
# Select relevant columns: metadata + year columns + Total
df_data = df_sorted[["OdName", "AreaName", "RegName", "DevName"] + year_cols + ["Total"]]

# Rename first column to blank (we'll put "Country" in first row)
df_data = df_data.rename(columns={"OdName": ""})

# Prepare first row with "Country" label in the first cell
first_row = pd.DataFrame([["Country"] + [""]*(df_data.shape[1]-1)], columns=df_data.columns)

# Combine first row with data
df_output = pd.concat([first_row, df_data], ignore_index=True)

#Sets up the display table with the first row showing “Country” in bold.
#Keeps all other metadata and yearly columns intact.


In [7]:
# Style function to bold first row, first cell
def bold_first_cell(s):
    df_styles = pd.DataFrame("", index=s.index, columns=s.columns)
    df_styles.iloc[0, 0] = "font-weight: bold"
    return df_styles

# Display styled table in Jupyter Notebook
pd.set_option("display.max_columns", None)
df_output.style.apply(bold_first_cell, axis=None)

#pd.set_option("display.max_columns", None) ensures all year columns are visible.
#The table now looks like a professional report with a bold “Country” label at the top.


,,AreaName,RegName,DevName,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
0,Country,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,741,828,1076,1028,1378,1170,713,858,1537,2212,2555,1999,2395,3326,4067,3697,3479,2978,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
2,Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,2,2,3,3,21,56,96,71,63,113,307,574,1264,1816,1602,1021,853,1450,1223,856,702,560,716,561,539,620,603,15699
3,Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,132,242,434,491,872,795,717,595,1106,2054,1842,2292,2389,2867,3418,3406,3072,3616,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
4,American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,1,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,6
5,Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,0,0,0,3,0,1,0,0,0,0,0,2,0,0,1,0,2,0,0,1,1,0,0,0,0,1,1,15
6,Angola,Africa,Middle Africa,Developing regions,1,3,6,6,4,3,5,5,11,6,8,23,26,22,8,26,38,27,58,49,70,169,168,165,268,295,184,106,76,62,61,39,70,45,2113
7,Antigua and Barbuda,Latin America and the Caribbean,Caribbean,Developing regions,0,0,0,0,42,52,51,61,34,51,31,59,61,44,18,23,21,27,19,12,13,25,17,25,14,24,32,15,32,38,27,37,51,25,981
8,Argentina,Latin America and the Caribbean,South America,Developing regions,368,426,626,241,237,196,213,519,374,538,678,763,1023,1022,366,380,396,391,349,318,427,635,865,1745,1591,1153,847,620,540,467,459,278,263,282,19596
9,Armenia,Asia,Western Asia,Developing regions,0,0,0,0,0,0,0,0,0,0,0,0,22,21,66,75,102,115,89,112,124,87,132,153,147,224,218,198,205,267,252,236,258,207,3310
